### Step 1: Optimising Zero Shot Prompts (Testing Pipeline: Phase 1 + Phase 2)


In [ ]:
import os, re, time, json
from pathlib import Path
from datetime import datetime
from collections import Counter

import pandas as pd
import numpy as np
import ollama
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

#### 1. Configuration

In [ ]:
# Step 1: Set paths
notebook_dir = Path.cwd()
DATA_DIR = notebook_dir

RESULTS_DIR = DATA_DIR / 'prompt_testing_results'
RESULTS_DIR.mkdir(exist_ok=True)


In [ ]:
# Step 2: Keep both target models in config.
# since I had one model missing locally I had an auto-skip later with RUN_MODELS.
MODELS = [
    ('llama3.1:8b', 'llama3.1_8b'),
    ('mixtral:8x7b', 'mixtral_8x7b'),
]

# We can add more models later by extending this list.


In [ ]:
# Sampling settings
SAMPLE_SIZE  = 50
N_BATCHES    = 3
RANDOM_STATE = 42 
TEMPERATURE  = 0.0

#### 2. Prompt Variants

In [ ]:
# 2. PROMPT VARIANTS
#
# We evaluate both phases in one run:
# - Phase 1 uses review title + review abstract
# - Phase 2 uses objectives_text + selection_criteria_text
#
# Shared paper fields:
#   {paper_title} {paper_abstract}
#
# Every prompt must end with: INCLUDE or EXCLUDE.


Phase 1: Review Title + Review Abstract


In [ ]:
PHASE1_PROMPT_VARIANTS = [

    # V1: Baseline
    {
        "name": "p1_v1_baseline",
        "system": None,
        "user": (
            "You are assisting with title/abstract screening for a Cochrane systematic review.\n\n"
            "=== REVIEW ===\n"
            "Title: {review_title}\n\n"
            "Abstract: {review_abstract}\n\n"
            "=== CANDIDATE PAPER ===\n"
            "Title: {paper_title}\n\n"
            "Abstract: {paper_abstract}\n\n"
            "Based on the information above, should this paper be included in the review?\n\n"
            "Respond with exactly one word: INCLUDE or EXCLUDE"
        ),
    },

    # V2: No role framing
    {
        "name": "p1_v2_no_role",
        "system": None,
        "user": (
            "=== REVIEW ===\n"
            "Title: {review_title}\n\n"
            "Abstract: {review_abstract}\n\n"
            "=== CANDIDATE PAPER ===\n"
            "Title: {paper_title}\n\n"
            "Abstract: {paper_abstract}\n\n"
            "Based on the information above, should this paper be included in the review?\n\n"
            "Respond with exactly one word: INCLUDE or EXCLUDE"
        ),
    },

    # V3: With system prompt
    {
        "name": "p1_v3_with_system_prompt",
        "system": (
            "You are an expert medical librarian assisting with title/abstract screening "
            "for Cochrane systematic reviews. Your task is to decide whether a candidate "
            "paper is relevant to a given review. Always respond with exactly one word: "
            "INCLUDE or EXCLUDE."
        ),
        "user": (
            "=== REVIEW ===\n"
            "Title: {review_title}\n\n"
            "Abstract: {review_abstract}\n\n"
            "=== CANDIDATE PAPER ===\n"
            "Title: {paper_title}\n\n"
            "Abstract: {paper_abstract}\n\n"
            "Should this paper be included in the review?\n\n"
            "Respond with exactly one word: INCLUDE or EXCLUDE"
        ),
    },

    # V4: Minimal wording
    {
        "name": "p1_v4_minimal",
        "system": None,
        "user": (
            "You are assisting with title/abstract screening for a Cochrane systematic review.\n\n"
            "Review title: {review_title}\n"
            "Review abstract: {review_abstract}\n\n"
            "Paper title: {paper_title}\n"
            "Paper abstract: {paper_abstract}\n\n"
            "Respond with exactly one word: INCLUDE or EXCLUDE"
        ),
    },

    # V5: Detailed wording
    {
        "name": "p1_v5_detailed",
        "system": None,
        "user": (
            "You are assisting with title/abstract screening for a Cochrane systematic review.\n\n"
            "Step 1 — Understand the review scope:\n"
            "  Title: {review_title}\n"
            "  Abstract: {review_abstract}\n\n"
            "Step 2 — Evaluate the candidate paper:\n"
            "  Title: {paper_title}\n"
            "  Abstract: {paper_abstract}\n\n"
            "Step 3 — Consider whether the paper:\n"
            "  - Studies the same population, condition, or disease as the review\n"
            "  - Evaluates a relevant intervention, exposure, or treatment\n"
            "  - Uses a study design relevant to the review (e.g. RCT, cohort)\n"
            "  - Reports outcomes relevant to the review question\n\n"
            "Step 4 — Make your decision.\n"
            "Respond with exactly one word: INCLUDE or EXCLUDE"
        ),
    },
]



Phase 2: Objectives + Selection Criteria (from Methodology extraction)


In [ ]:
PHASE2_PROMPT_VARIANTS = [

    # V1: Baseline
    {
        "name": "p2_v1_baseline_obj_sel",
        "system": None,
        "user": (
            "You are assisting with title/abstract screening for a Cochrane systematic review.\n\n"
            "=== REVIEW CONTEXT ===\n"
            "Objectives: {objectives_text}\n\n"
            "Selection criteria: {selection_criteria_text}\n\n"
            "=== CANDIDATE PAPER ===\n"
            "Title: {paper_title}\n\n"
            "Abstract: {paper_abstract}\n\n"
            "Using only the objectives and selection criteria above, should this paper be included in the review?\n\n"
            "Respond with exactly one word: INCLUDE or EXCLUDE"
        ),
    },

    # V2: No role framing
    {
        "name": "p2_v2_no_role_obj_sel",
        "system": None,
        "user": (
            "=== REVIEW CONTEXT ===\n"
            "Objectives: {objectives_text}\n\n"
            "Selection criteria: {selection_criteria_text}\n\n"
            "=== CANDIDATE PAPER ===\n"
            "Title: {paper_title}\n\n"
            "Abstract: {paper_abstract}\n\n"
            "Using only the objectives and selection criteria above, should this paper be included in the review?\n\n"
            "Respond with exactly one word: INCLUDE or EXCLUDE"
        ),
    },

    # V3: With system prompt
    {
        "name": "p2_v3_with_system_prompt_obj_sel",
        "system": (
            "You are an expert medical librarian assisting with title/abstract screening "
            "for Cochrane systematic reviews. Use only the review objectives and "
            "selection criteria in the user prompt. Always respond with exactly one word: "
            "INCLUDE or EXCLUDE."
        ),
        "user": (
            "=== REVIEW CONTEXT ===\n"
            "Objectives: {objectives_text}\n\n"
            "Selection criteria: {selection_criteria_text}\n\n"
            "=== CANDIDATE PAPER ===\n"
            "Title: {paper_title}\n\n"
            "Abstract: {paper_abstract}\n\n"
            "Should this paper be included in the review?\n\n"
            "Respond with exactly one word: INCLUDE or EXCLUDE"
        ),
    },

    # V4: Minimal wording
    {
        "name": "p2_v4_minimal_obj_sel",
        "system": None,
        "user": (
            "Objectives: {objectives_text}\n\n"
            "Selection criteria: {selection_criteria_text}\n\n"
            "Paper title: {paper_title}\n"
            "Paper abstract: {paper_abstract}\n\n"
            "Use only the objectives and selection criteria. "
            "Respond with exactly one word: INCLUDE or EXCLUDE"
        ),
    },

    # V5: Detailed wording
    {
        "name": "p2_v5_detailed_obj_sel",
        "system": None,
        "user": (
            "You are assisting with title/abstract screening for a Cochrane systematic review.\n\n"
            "Step 1 — Understand the review scope from these fields only:\n"
            "  Objectives: {objectives_text}\n"
            "  Selection criteria: {selection_criteria_text}\n\n"
            "Step 2 — Evaluate the candidate paper:\n"
            "  Title: {paper_title}\n"
            "  Abstract: {paper_abstract}\n\n"
            "Step 3 — Decide whether the paper matches the review scope.\n"
            "Step 4 — Give one-word output.\n"
            "Respond with exactly one word: INCLUDE or EXCLUDE"
        ),
    },
]

# One combined prompt list so the rest of Version 2 can run almost unchanged.
PROMPT_VARIANTS = (
    [{**v, 'phase': 'phase1_title_abstract'} for v in PHASE1_PROMPT_VARIANTS] +
    [{**v, 'phase': 'phase2_objectives_selection'} for v in PHASE2_PROMPT_VARIANTS]
)

print(f"Prompt variants ready: {len(PROMPT_VARIANTS)} total")


#### 3. Helper Functions

In [ ]:
def create_prompt(row, template):
    return template.format(
        review_title            = str(row.get('review_title', ''))[:500],
        review_abstract         = str(row.get('review_abstract', ''))[:2000],
        objectives_text         = str(row.get('objectives_text', ''))[:2500],
        selection_criteria_text = str(row.get('selection_criteria_text', ''))[:2500],
        paper_title             = str(row.get('paper_title', ''))[:300],
        paper_abstract          = str(row.get('paper_abstract', ''))[:2000],
    )


In [ ]:
def extract_decision(response: str) -> int:
    if not response or not response.strip():
        return -1
    up = response.upper()
    m = re.search(r'DECISION:\s*(INCLUDE|EXCLUDE)', up)
    if m:
        return 1 if m.group(1) == 'INCLUDE' else 0
    for line in reversed(up.strip().split('\n')[-5:]):
        line = line.strip()
        if line == 'INCLUDE': return 1
        if line == 'EXCLUDE': return 0
    inc, exc = up.rfind('INCLUDE'), up.rfind('EXCLUDE')
    if inc > exc: return 1
    if exc > inc: return 0
    return -1

In [ ]:
# identical to 07_evaluate_llms.ipynb, with one addition: optional `system_prompt` parameter passed to ollama.generate().
def llm_call(model, prompt, temperature=TEMPERATURE, num_predict=2048,
             system_prompt=None):
    t0 = time.time()
    kwargs = dict(
        model=model,
        prompt=prompt,
        options={'temperature': temperature, 'num_predict': num_predict},
    )
    if system_prompt:
        kwargs['system'] = system_prompt
    resp = ollama.generate(**kwargs)
    return resp.get('response', ''), time.time() - t0


In [ ]:
# identical to 07_evaluate_llms.ipynb, with one addition: `invalid` count (predictions that could not be parsed) in returned dict.
def score(labels, preds):
    mask = [p != -1 for p in preds]
    yt = [l for l, m in zip(labels, mask) if m]
    yp = [p for p, m in zip(preds,  mask) if m]
    n  = sum(mask)
    if n == 0:
        return dict(valid=0, invalid=len(preds), acc=0, prec=0, rec=0, f1=0)
    return dict(
        valid   = n,
        invalid = len(preds) - n,
        acc     = round(accuracy_score(yt, yp), 4),
        prec    = round(precision_score(yt, yp, zero_division=0), 4),
        rec     = round(recall_score(yt, yp,    zero_division=0), 4),
        f1      = round(f1_score(yt, yp,        zero_division=0), 4),
    )


In [ ]:
def print_metrics(name, labels, preds):
    """Identical to 07_evaluate_llms.ipynb."""
    s = score(labels, preds)
    print(f"\n{'='*60}")
    print(f"{name}  ({s['valid']}/{len(labels)} valid)")
    print(f"{'='*60}")
    print(f"  Acc={s['acc']:.3f}  Prec={s['prec']:.3f}  "
          f"Rec={s['rec']:.3f}  F1={s['f1']:.3f}")
    vt = [l for l, p in zip(labels, preds) if p != -1]
    vp = [p for p in preds if p != -1]
    if len(vt) > 0:
        cm = confusion_matrix(vt, vp)
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
            print(f"  TP={tp}  FP={fp}  FN={fn}  TN={tn}")
    return s

#### 4. Load Data and Attach Phase 2 Context


In [ ]:
# Step 4: Check which input files are available.
ground_truth_csv    = DATA_DIR / 'ground_truth_validation_dataset.csv'

phase2_context_path = DATA_DIR / 'review_phase2_context.csv'
full_extraction_path = DATA_DIR / 'review_criteria_extraction.csv' # what is this? 




In [ ]:
#Load validation data.
ground_truth = pd.read_csv(ground_truth_csv, low_memory=False) 


#Load objectives + selection criteria extracted from methodology notebook.

criteria_df = pd.read_csv(
        phase2_context_path,
        usecols=['review_id', 'objectives_text', 'selection_criteria_text'],
        low_memory=False,
    )

criteria_df = (
    criteria_df
    .drop_duplicates(subset=['review_id'])
    .assign(review_id=lambda d: d['review_id'].astype(str).str.upper().str.strip())
)



In [ ]:
ground_truth.head(3)


In [ ]:
#Major issue here is that since I had to use pdf's to get out the methodology, and that we don't 
#the full validation set, then I had to map each paper to the correct review document and attach objectives/selection criteria.
ground_truth = ground_truth.copy()
ground_truth['review_id'] = ground_truth['review_doi'].astype(str).str.extract(r'(CD\d{6})', expand=False)

merged_data = ground_truth.merge(
    criteria_df,
    on='review_id',
    how='left',
    validate='m:1',
)

phase_ready_data = merged_data[
    merged_data['objectives_text'].fillna('').str.strip().ne('')
    & merged_data['selection_criteria_text'].fillna('').str.strip().ne('')
].copy()

# Keep only reviews that have both INCLUDE and EXCLUDE papers.
review_labels = phase_ready_data.groupby('review_doi')['label'].agg(['sum', 'count']).reset_index()
review_labels.columns = ['review_doi', 'n_included', 'n_total']
review_labels['n_excluded'] = review_labels['n_total'] - review_labels['n_included']
reviews_with_both = review_labels[
    (review_labels['n_included'] > 0) & (review_labels['n_excluded'] > 0)
]

eval_data = phase_ready_data[
    phase_ready_data['review_doi'].isin(reviews_with_both['review_doi'])
].copy()

print(f"Rows after merge: {len(merged_data):,}")
print(f"Rows with phase 2 context: {len(phase_ready_data):,}")
print(f"Eligible for evaluation: {len(eval_data):,} ({eval_data['review_doi'].nunique()} reviews)")


In [ ]:
eval_data[['review_doi', 'review_id', 'paper_title', 'label', 'objectives_text', 'selection_criteria_text']].head(5)


#### 5. Build 3 Non-Overlapping Batches


In [ ]:
# Step 8: Build class-balanced batch targets based on the full eligible data.
overall_inc_rate = eval_data['label'].mean()
n_inc_per_batch = round(SAMPLE_SIZE * overall_inc_rate)
n_exc_per_batch = SAMPLE_SIZE - n_inc_per_batch

print(f"Overall inclusion rate: {overall_inc_rate:.1%}")
print(f"Per batch target: {n_inc_per_batch} INC / {n_exc_per_batch} EXC")


In [ ]:
inc_pool = eval_data[eval_data['label'] == 1].sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
exc_pool = eval_data[eval_data['label'] == 0].sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

batches = []
for i in range(N_BATCHES):
    inc_batch = inc_pool.iloc[i * n_inc_per_batch : (i + 1) * n_inc_per_batch]
    exc_batch = exc_pool.iloc[i * n_exc_per_batch : (i + 1) * n_exc_per_batch]
    batch = pd.concat([inc_batch, exc_batch]).sample(frac=1, random_state=RANDOM_STATE + i).reset_index(drop=True)
    batches.append(batch)
    print(
        f"Batch {i+1}: {len(batch)} papers | "
        f"{(batch['label']==1).sum()} INC / {(batch['label']==0).sum()} EXC | "
        f"{batch['review_doi'].nunique()} reviews"
    )


#### 6. Run Evaluation

In [ ]:
#SKIPS MODEL IF MISSING (MISTRAL)
available = [m.model for m in ollama.list().models]
RUN_MODELS = [(m, s) for (m, s) in MODELS if any(m in a for a in available)]
missing = [m for (m, _) in MODELS if not any(m in a for a in available)]

if missing:
    print(f"Skipping missing models: {missing}")

print(f"Running models: {[m for m, _ in RUN_MODELS]}")

if not RUN_MODELS:
    raise RuntimeError('No requested models are installed in Ollama.')


In [ ]:
all_rows = []
total_runs = len(batches) * len(RUN_MODELS) * len(PROMPT_VARIANTS)
run_idx = 0


In [ ]:
for batch_idx, test_subset in enumerate(batches):
    labels = test_subset['label'].tolist()

    for model_name, model_slug in RUN_MODELS:
        for variant in PROMPT_VARIANTS:
            run_idx += 1
            vname = variant['name']
            phase = variant.get('phase', 'unknown_phase')
            print(f"\n{'='*60}")
            print(f"[{run_idx}/{total_runs}]  batch={batch_idx+1}  model={model_name}  phase={phase}  prompt={vname}")
            print(f"{'='*60}")

            preds, times, raw_responses = [], [], []

            for _, row in tqdm(test_subset.iterrows(), total=len(test_subset),
                               desc=f"B{batch_idx+1} | {model_slug} | {vname}"):
                user_prompt = create_prompt(row, variant['user'])
                system_prompt = variant.get('system')

                try:
                    resp, elapsed = llm_call(model_name, user_prompt, system_prompt=system_prompt)
                    pred = extract_decision(resp)
                except Exception as e:
                    resp, pred, elapsed = str(e), -1, 0.0

                preds.append(pred)
                times.append(elapsed)
                raw_responses.append(resp)

            m = print_metrics(f"B{batch_idx+1} | {model_name} | {phase} | {vname}", labels, preds)
            m['time_min'] = round(sum(times) / 60, 2)

            all_rows.append({
                'batch': batch_idx + 1,
                'phase': phase,
                'model': model_name,
                'prompt': vname,
                'has_system': variant.get('system') is not None,
                **m,
            })

            # Save per-run raw predictions
            run_df = test_subset[['review_doi', 'review_id', 'paper_title', 'label']].copy()
            run_df['phase'] = phase
            run_df['model'] = model_name
            run_df['prompt'] = vname
            run_df['prediction'] = preds
            run_df['response'] = raw_responses
            run_df['time_sec'] = [round(t, 2) for t in times]

            ts = datetime.now().strftime('%Y%m%d_%H%M%S')
            run_df.to_csv(
                RESULTS_DIR / f'run_B{batch_idx+1}_{model_slug}_{vname}_{ts}.csv',
                index=False
            )


#### 7. Results Table

In [ ]:
results_df = pd.DataFrame(all_rows)
display_cols = ['batch', 'phase', 'model', 'prompt', 'has_system', 'acc', 'prec', 'rec', 'f1', 'invalid', 'time_min']

print("RESULTS: ALL BATCHES x PHASES x PROMPTS x MODELS (sorted by F1)")
print("=" * 90)
print(results_df.sort_values('f1', ascending=False)[display_cols].to_string(index=False))

print("\nMEAN METRICS ACROSS BATCHES (grouped by phase x model x prompt)")
print("=" * 90)
agg = (
    results_df
    .groupby(['phase', 'model', 'prompt', 'has_system'])[['acc', 'prec', 'rec', 'f1']]
    .mean()
    .round(3)
    .reset_index()
)
print(agg.sort_values('f1', ascending=False).to_string(index=False))

print("\nF1 PIVOT TABLE (rows=phase+prompt, cols=model)")
print("=" * 90)
print(agg.pivot(index=['phase', 'prompt'], columns='model', values='f1').round(3).to_string())

print("\nRECALL PIVOT TABLE (rows=phase+prompt, cols=model)")
print("=" * 90)
print(agg.pivot(index=['phase', 'prompt'], columns='model', values='rec').round(3).to_string())


In [ ]:
# Save the full run results as one CSV file.
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
out_path = RESULTS_DIR / f'prompt_comparison_{ts}.csv'
results_df.sort_values(['phase', 'model', 'prompt', 'batch']).to_csv(out_path, index=False)
print(f"\nFull results saved to {out_path}")
